Live Data Ingestion Setup

This section initializes the environment and connects to the NYC Open Data API.

Instead of using a static CSV file, this project pulls live 311 service request data directly from the source.

The query dynamically filters records from 2024 to the current date, allowing the dataset to stay up-to-date each time the script is run.

URL encoding (%20) is used to ensure the API query is properly formatted.

This pipeline is designed to scale with increasing data volume and can be rerun to incorporate newly available records.

Due to API rate limits, requests are throttled and retried using a backoff strategy to ensure reliable data retrieval.

In [1]:
from datetime import datetime
import pandas as pd
from urllib.parse import quote
import time  # 👈 ADD HERE

today = datetime.today().strftime('%Y-%m-%d')

Chunked Data Loading and Processing

Due to the large size of the dataset, the data is processed in chunks of 100,000 rows at a time.

This approach prevents memory overload and allows efficient handling of large-scale data.

Within each chunk:

Column names are standardized (lowercase, no spaces)
The created_date field is converted to a datetime format
Only relevant columns are selected for analysis

Each processed chunk is stored for later combination.

In [2]:
start = "2024-01-01"
end = "2026-04-14"

dates = pd.date_range(start=start, end=end, freq='MS')

date_ranges = []

for i in range(len(dates)):
    start_date = dates[i]
    
    if i + 1 < len(dates):
        end_date = dates[i + 1] - pd.Timedelta(days=1)
    else:
        end_date = pd.to_datetime(end)

    date_ranges.append((
        start_date.strftime('%Y-%m-%d'),
        end_date.strftime('%Y-%m-%d')
    ))

date_ranges[:5]

[('2024-01-01', '2024-01-31'),
 ('2024-02-01', '2024-02-29'),
 ('2024-03-01', '2024-03-31'),
 ('2024-04-01', '2024-04-30'),
 ('2024-05-01', '2024-05-31')]

Combining Processed Chunks

After all chunks have been processed, they are combined into a single DataFrame.

This creates a unified dataset containing all filtered and cleaned records retrieved from the API.

In [3]:
import time

all_chunks = []

for start_date, end_date in date_ranges:
    print(f"\nPulling data from {start_date} to {end_date}")

    query = f"created_date between '{start_date}T00:00:00' and '{end_date}T23:59:59'"
    encoded_query = quote(query)

    file_path = (
        "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?"
        f"$limit=500000&$where={encoded_query}"
    )

    success = False
    attempts = 0

    while not success and attempts < 3:
        try:
            for i, chunk in enumerate(pd.read_csv(
                file_path,
                chunksize=100000,
                dtype={"taxi_company_borough": "string"},
                low_memory=False
            )):
                print(f"  Processing chunk {i+1}")

                chunk.columns = chunk.columns.str.strip().str.lower().str.replace(" ", "_")
                chunk['created_date'] = pd.to_datetime(chunk['created_date'], errors='coerce')

                filtered = chunk[['unique_key', 'created_date', 'complaint_type', 'descriptor', 'borough']]
                all_chunks.append(filtered)

                # 🔥 RATE LIMIT BETWEEN CHUNKS
                time.sleep(1)

            success = True

        except Exception as e:
            attempts += 1
            print(f"⚠️ Error occurred. Retrying... ({attempts}/3)")
            print(e)

            # 🔥 BACKOFF STRATEGY (wait longer after failure)
            time.sleep(5 * attempts)

    # 🔥 RATE LIMIT BETWEEN MONTHS
    print("⏳ Waiting before next request...\n")
    time.sleep(2)


Pulling data from 2024-01-01 to 2024-01-31
  Processing chunk 1
  Processing chunk 2
  Processing chunk 3
⏳ Waiting before next request...


Pulling data from 2024-02-01 to 2024-02-29
  Processing chunk 1
  Processing chunk 2
  Processing chunk 3
⏳ Waiting before next request...


Pulling data from 2024-03-01 to 2024-03-31
  Processing chunk 1
  Processing chunk 2
  Processing chunk 3
⏳ Waiting before next request...


Pulling data from 2024-04-01 to 2024-04-30
  Processing chunk 1
  Processing chunk 2
  Processing chunk 3
⏳ Waiting before next request...


Pulling data from 2024-05-01 to 2024-05-31
  Processing chunk 1
  Processing chunk 2
  Processing chunk 3
⏳ Waiting before next request...


Pulling data from 2024-06-01 to 2024-06-30
  Processing chunk 1
  Processing chunk 2
  Processing chunk 3
  Processing chunk 4
⏳ Waiting before next request...


Pulling data from 2024-07-01 to 2024-07-31
  Processing chunk 1
  Processing chunk 2
  Processing chunk 3
⏳ Waiting before next requ

Data Consolidation

All processed data chunks are combined into a single DataFrame.

This step creates a unified dataset from individually processed segments, enabling full-dataset analysis.

The initial shape of the dataset is printed to establish a baseline before any data quality checks or

In [4]:
df = pd.concat(all_chunks, ignore_index=True)

print("Shape BEFORE cleaning:", df.shape)

Shape BEFORE cleaning: (8275986, 5)


## Data Validation (Pre-Cleaning) ⭐

Before modifying the dataset, a series of validation checks are performed to assess data quality and identify potential issues.

These checks include:

Duplicate detection based on unique_key
Null value analysis across all columns
Date range validation to confirm expected time coverage
Record distribution by year to identify inconsistencies or anomalies

Performing validation prior to cleaning ensures that data issues are understood and documented before any transformations are applied.

In [5]:
print("=== DATA VALIDATION REPORT ===\n")

# 1. Duplicate Check BEFORE removal
duplicate_count = df.duplicated(subset=['unique_key']).sum()
print("Duplicate rows (before cleaning):", duplicate_count)

# 2. Null Check
print("\nNull Values by Column:")
print(df.isnull().sum())

# 3. Date Range Check
print("\nDate Range:")
print("Start:", df['created_date'].min())
print("End:  ", df['created_date'].max())

# 4. Records by Year (distribution check)
print("\nRecords by Year:")
print(df['created_date'].dt.year.value_counts().sort_index())

=== DATA VALIDATION REPORT ===

Duplicate rows (before cleaning): 0

Null Values by Column:
unique_key             0
created_date           0
complaint_type         0
descriptor        209940
borough                0
dtype: int64

Date Range:
Start: 2024-01-01 00:00:00
End:   2026-04-14 23:59:57

Records by Year:
created_date
2024    3456770
2025    3654957
2026    1164259
Name: count, dtype: int64


## Data Cleaning (Deduplication)

Duplicate records are removed using the unique_key field.

This step ensures that each service request is represented only once in the dataset, improving data accuracy and preventing inflated counts in downstream analysis.

In [6]:
df = df.drop_duplicates(subset=['unique_key'])

print("\nShape AFTER removing duplicates:", df.shape)


Shape AFTER removing duplicates: (8275986, 5)


Final Output & Logging
Final Dataset Export and Refresh Summary

The cleaned dataset is saved as a validated master CSV file for downstream use.

A summary of the pipeline execution is printed, including:

Final row count after cleaning
Confirmation of successful refresh
Timestamp of the latest update
Dataset shape before and after duplicate removal

This provides transparency into how the dataset was transformed during the refresh process.

In [7]:
# The output path shown below is configured for the local development environment and may need to be updated for use on another machine.
output_path = "/Users/newworld/Downloads/311_master_dataset_validated.csv"
df.to_csv(output_path, index=False)

print("Refresh complete ✅")
print("Final Rows:", df.shape[0])
print("Last updated:", today)

Refresh complete ✅
Final Rows: 8275986
Last updated: 2026-04-20
